# DeFi Liquidity Pool Dashboard — FinTech 590

Interactive Streamlit dashboard for the Uniswap V3 pipeline outputs.

**Run order:** `defi_pipeline.ipynb` → `historical_data.ipynb` → `database.ipynb` → `onchain_data.ipynb` → this notebook.

**Inputs:**
- `data/top_pools.parquet` — current pool snapshot (TVL, volume, verification)
- `data/pool_history.parquet` — daily TVL, APY, IL per pool (May 2021 → present)
- `data/pool_onchain.parquet` — live on-chain state (price, tick, liquidity)

**To launch the dashboard:** run all cells top to bottom. The last cell (Cell 10) converts this notebook to a temporary script and opens Streamlit at http://localhost:8501.

In [1]:
# Cell 0 — Install dependencies
import subprocess, sys
for pkg in ["streamlit", "plotly", "pandas", "pyarrow"]:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=False)

In [2]:
# Cell 1 — Imports and config
import pathlib
import warnings
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st

warnings.filterwarnings("ignore")

st.set_page_config(
    page_title="DeFi Liquidity Pool Dashboard",
    page_icon="💧",
    layout="wide",
)

DATA_DIR = pathlib.Path("data")

CHAIN_COLORS = {
    "Ethereum": "#627EEA",
    "Arbitrum": "#28A0F0",
    "Optimism": "#FF0420",
    "Base":     "#0052FF",
    "Polygon":  "#8247E5",
}

2026-04-05 00:22:02.050 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [3]:
# Cell 2 — Data loading
@st.cache_data
def load_data():
    pools   = pd.read_parquet(DATA_DIR / "top_pools.parquet")
    history = pd.read_parquet(DATA_DIR / "pool_history.parquet")
    onchain = pd.read_parquet(DATA_DIR / "pool_onchain.parquet")

    pools["label"] = (
        pools["token0"] + "/" + pools["token1"]
        + " " + (pools["fee_tier"] / 1e4).map("{:.2f}%".format)
        + " (" + pools["chain"] + ")"
    )
    history["label"] = (
        history["token0"] + "/" + history["token1"]
        + " " + (history["fee_tier"] / 1e4).map("{:.2f}%".format)
    )
    onchain["label"] = (
        onchain["token0"] + "/" + onchain["token1"]
        + " " + (onchain["fee"] / 1e4).map("{:.2f}%".format)
        + " (" + onchain["chain"] + ")"
    )

    pools["fee_rate"]       = pools["fee_tier"] / 1e6
    pools["cap_efficiency"] = pools["volume_usd"] / pools["tvl_usd"]
    pools["fee_revenue"]    = pools["volume_usd"] * pools["fee_rate"]

    history["date"]      = pd.to_datetime(history["date"])
    onchain["liquidity"] = onchain["liquidity"].astype(float)

    return pools, history, onchain


pools, history, onchain = load_data()

2026-04-05 00:22:02.066 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-04-05 00:22:02.068 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-04-05 00:22:02.069 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [4]:
# Cell 3 — Sidebar filters
st.sidebar.title("Filters")

all_chains     = sorted(pools["chain"].unique())
selected_chains = st.sidebar.multiselect("Chains", all_chains, default=all_chains)
search          = st.sidebar.text_input("Search pool (e.g. USDC, WETH)", "")
history_days    = st.sidebar.slider("History window (days)", 30, 365 * 5, 365, step=30)

# Apply filters
pools_f = pools[pools["chain"].isin(selected_chains)].copy()
if search:
    mask    = pools_f["token0"].str.contains(search, case=False) | pools_f["token1"].str.contains(search, case=False)
    pools_f = pools_f[mask]

history_f      = history[history["chain"].isin(selected_chains)].copy()
cutoff         = history_f["date"].max() - pd.Timedelta(days=history_days)
history_window = history_f[history_f["date"] >= cutoff]

onchain_f = onchain[onchain["chain"].isin(selected_chains)].copy()

2026-04-05 00:22:02.263 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:03.087 
  command:

    streamlit run C:\Users\Zhou\anaconda3\envs\data_wrangling\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-04-05 00:22:03.088 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:03.090 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:03.092 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:03.093 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:03.094 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:03.095 Thre

In [5]:
# Cell 4 — Header and KPI metrics
st.title("💧 DeFi Liquidity Pool Dashboard")
st.caption(
    f"Uniswap V3 \u00b7 {', '.join(selected_chains)} \u00b7 "
    f"Pool snapshot: {pools['fetched_at'].max().date() if 'fetched_at' in pools.columns else 'N/A'} \u00b7 "
    f"On-chain data: {onchain['fetched_at'].iloc[0] if 'fetched_at' in onchain.columns else 'N/A'}"
)

total_tvl  = pools_f["tvl_usd"].sum()
total_vol  = pools_f["volume_usd"].sum()
total_fees = pools_f["fee_revenue"].sum()
median_apy = history_window.groupby("address")["apy_base"].median().median()
n_pools    = len(pools_f)

k1, k2, k3, k4, k5 = st.columns(5)
k1.metric("Total TVL",          f"${total_tvl/1e9:.2f}B")
k2.metric("24h Volume",         f"${total_vol/1e6:.0f}M")
k3.metric("Est. Daily Fees",    f"${total_fees/1e3:.0f}K")
k4.metric("Median APY (window)", f"{median_apy:.1f}%" if pd.notna(median_apy) else "N/A")
k5.metric("Pools",              n_pools)

st.divider()

2026-04-05 00:22:03.140 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:03.141 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:03.142 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:03.143 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:03.144 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:03.145 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:03.151 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:03.152 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator()

In [6]:
# Cell 5 — Section 1: Current Pool Snapshot
st.subheader("1 \u00b7 Current Pool Snapshot")

tab_tvl, tab_vol, tab_efficiency = st.tabs(["TVL", "Volume & Fee Revenue", "Capital Efficiency"])

with tab_tvl:
    top15 = pools_f.sort_values("tvl_usd", ascending=False).head(15)
    fig = px.bar(
        top15.sort_values("tvl_usd"),
        x="tvl_usd", y="label", orientation="h",
        color="chain", color_discrete_map=CHAIN_COLORS,
        labels={"tvl_usd": "TVL (USD)", "label": ""},
        title="TVL by Pool (Top 15)",
    )
    fig.update_xaxes(tickprefix="$", tickformat=",.0f")
    fig.update_layout(height=500)
    st.plotly_chart(fig, use_container_width=True)

with tab_vol:
    top15 = pools_f.sort_values("volume_usd", ascending=False).head(15)
    fig = px.bar(
        top15.sort_values("fee_revenue"),
        x="fee_revenue", y="label", orientation="h",
        color="chain", color_discrete_map=CHAIN_COLORS,
        labels={"fee_revenue": "Est. Daily Fee Revenue (USD)", "label": ""},
        title="Estimated Daily Fee Revenue by Pool (Top 15)",
    )
    fig.update_xaxes(tickprefix="$", tickformat=",.0f")
    fig.update_layout(height=500)
    st.plotly_chart(fig, use_container_width=True)

with tab_efficiency:
    fig = px.scatter(
        pools_f.dropna(subset=["cap_efficiency"]),
        x="tvl_usd", y="volume_usd",
        size="fee_revenue", color="chain",
        color_discrete_map=CHAIN_COLORS,
        hover_name="label",
        log_x=True, log_y=True,
        labels={"tvl_usd": "TVL (USD, log)", "volume_usd": "24h Volume (USD, log)"},
        title="TVL vs. Volume \u2014 Capital Efficiency (bubble = fee revenue)",
    )
    fig.update_xaxes(tickprefix="$")
    fig.update_yaxes(tickprefix="$")
    st.plotly_chart(fig, use_container_width=True)

with st.expander("Full pool table"):
    display_cols = [c for c in ["label", "chain", "tvl_usd", "volume_usd", "cap_efficiency", "fee_revenue"] if c in pools_f.columns]
    st.dataframe(
        pools_f[display_cols].sort_values("tvl_usd", ascending=False).reset_index(drop=True)
        .style.format({"tvl_usd": "${:,.0f}", "volume_usd": "${:,.0f}", "cap_efficiency": "{:.3f}", "fee_revenue": "${:,.0f}"}),
        use_container_width=True,
    )

st.divider()

2026-04-05 00:22:03.188 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:03.189 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:03.189 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:03.190 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:03.191 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:03.192 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:03.192 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:03.446 Please replace `use_container_width` with `width`.

`use_container_width` will be removed afte

DeltaGenerator()

In [7]:
# Cell 6 — Section 2: Historical TVL Trends
st.subheader("2 \u00b7 Historical TVL Trends")

tab_total, tab_chain, tab_indexed = st.tabs(["Total", "By Chain", "Indexed Growth"])

with tab_total:
    daily_total = history_window.groupby("date")["tvl_usd"].sum().reset_index()
    fig = px.area(
        daily_total, x="date", y="tvl_usd",
        labels={"tvl_usd": "Total TVL (USD)", "date": ""},
        title="Total TVL Over Time \u2014 All Tracked Pools",
    )
    fig.update_yaxes(tickprefix="$", tickformat=".2s")
    fig.update_layout(height=400)
    st.plotly_chart(fig, use_container_width=True)

with tab_chain:
    daily_chain = history_window.groupby(["date", "chain"])["tvl_usd"].sum().reset_index()
    fig = px.line(
        daily_chain, x="date", y="tvl_usd", color="chain",
        color_discrete_map=CHAIN_COLORS, log_y=True,
        labels={"tvl_usd": "TVL (USD, log scale)", "date": "", "chain": "Chain"},
        title="TVL Over Time by Chain (Log Scale)",
    )
    fig.update_yaxes(tickprefix="$", tickformat=".2s")
    fig.update_layout(height=400)
    st.plotly_chart(fig, use_container_width=True)

with tab_indexed:
    daily_chain = history_window.groupby(["date", "chain"])["tvl_usd"].sum().reset_index()
    traces = []
    for chain, grp in daily_chain.groupby("chain"):
        grp = grp.sort_values("date").copy()
        first = grp["tvl_usd"].iloc[0]
        if first > 0:
            grp["indexed"] = grp["tvl_usd"] / first * 100
            traces.append(grp.assign(chain=chain))
    if traces:
        indexed_df = pd.concat(traces)
        fig = px.line(
            indexed_df, x="date", y="indexed", color="chain",
            color_discrete_map=CHAIN_COLORS,
            labels={"indexed": "TVL Index (start = 100)", "date": "", "chain": "Chain"},
            title="TVL Growth Indexed to Window Start",
        )
        fig.add_hline(y=100, line_dash="dash", line_color="gray", opacity=0.5)
        fig.update_layout(height=400)
        st.plotly_chart(fig, use_container_width=True)

st.divider()

2026-04-05 00:22:04.527 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:04.527 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:04.528 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:04.529 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:04.529 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:04.530 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:04.531 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:04.568 Please replace `use_container_width` with `width`.

`use_container_width` will be removed afte

DeltaGenerator()

In [8]:
# Cell 7 — Section 3: APY Analysis
st.subheader("3 \u00b7 APY Analysis")

tab_median, tab_vol_apy, tab_pool_ts = st.tabs(["Median APY (window)", "APY Volatility", "Pool APY Over Time"])

with tab_median:
    median_apy_df = (
        history_window.groupby(["address", "label"])["apy_base"]
        .median().reset_index()
        .sort_values("apy_base", ascending=False).dropna().head(15)
    )
    fig = px.bar(
        median_apy_df.sort_values("apy_base"),
        x="apy_base", y="label", orientation="h",
        color="apy_base", color_continuous_scale="Greens",
        labels={"apy_base": "Median Base APY (%)", "label": ""},
        title=f"Top 15 Pools by Median Base APY (Last {history_days} Days)",
    )
    fig.update_xaxes(ticksuffix="%")
    fig.update_layout(height=500, coloraxis_showscale=False)
    st.plotly_chart(fig, use_container_width=True)

with tab_vol_apy:
    apy_vol_df = (
        history_window.groupby(["address", "label"])["apy_base"]
        .std().reset_index()
        .rename(columns={"apy_base": "apy_std"})
        .dropna().sort_values("apy_std", ascending=False).head(15)
    )
    fig = px.bar(
        apy_vol_df.sort_values("apy_std"),
        x="apy_std", y="label", orientation="h",
        color="apy_std", color_continuous_scale="Reds",
        labels={"apy_std": "APY Std Dev (%)", "label": ""},
        title=f"Top 15 Pools by APY Volatility (Last {history_days} Days)",
    )
    fig.update_xaxes(ticksuffix="%")
    fig.update_layout(height=500, coloraxis_showscale=False)
    st.plotly_chart(fig, use_container_width=True)

with tab_pool_ts:
    pool_options  = sorted(history_window["label"].dropna().unique())
    default_picks = pool_options[:3] if len(pool_options) >= 3 else pool_options
    selected_pools = st.multiselect("Select pools", pool_options, default=default_picks, key="apy_pools")
    if selected_pools:
        ts_data = history_window[history_window["label"].isin(selected_pools)]
        fig = px.line(
            ts_data, x="date", y="apy_base", color="label",
            labels={"apy_base": "Base APY (%)", "date": "", "label": "Pool"},
            title="Base APY Over Time \u2014 Selected Pools",
        )
        fig.update_yaxes(ticksuffix="%")
        fig.update_layout(height=400)
        st.plotly_chart(fig, use_container_width=True)

st.divider()

2026-04-05 00:22:04.787 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:04.788 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:04.789 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:04.790 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:04.790 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:04.791 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:04.792 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:04.858 Please replace `use_container_width` with `width`.

`use_container_width` will be removed afte

DeltaGenerator()

In [9]:
# Cell 8 — Section 4: Risk Metrics
st.subheader("4 \u00b7 Risk Metrics")

tab_il, tab_concentration = st.tabs(["Impermanent Loss", "TVL Concentration"])

with tab_il:
    il_data  = history_window.dropna(subset=["il_7d"])
    daily_il = il_data.groupby("date")["il_7d"].median().reset_index()
    fig = px.area(
        daily_il, x="date", y="il_7d",
        labels={"il_7d": "Median 7-Day IL (%)", "date": ""},
        title="Median Impermanent Loss Over Time (All Pools)",
        color_discrete_sequence=["tomato"],
    )
    fig.update_yaxes(tickformat=".3f", ticksuffix="%")
    fig.update_layout(height=400)
    st.plotly_chart(fig, use_container_width=True)

with tab_concentration:
    tvl_sorted = pools_f.sort_values("tvl_usd", ascending=False).reset_index(drop=True)
    top5_pct   = tvl_sorted.head(5)["tvl_usd"].sum() / tvl_sorted["tvl_usd"].sum() * 100
    fig = px.bar(
        tvl_sorted.head(15),
        x="label", y="tvl_usd",
        color="chain", color_discrete_map=CHAIN_COLORS,
        labels={"tvl_usd": "TVL (USD)", "label": ""},
        title=f"TVL Concentration \u2014 Top 5 pools hold {top5_pct:.1f}% of total TVL",
    )
    fig.update_yaxes(tickprefix="$", tickformat=",.0f")
    fig.update_xaxes(tickangle=45)
    fig.update_layout(height=450)
    st.plotly_chart(fig, use_container_width=True)

st.divider()

2026-04-05 00:22:05.025 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:05.026 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:05.026 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:05.027 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:05.028 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:05.030 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:05.075 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.


DeltaGenerator()

In [10]:
# Cell 9 — Section 5: On-Chain Live State
st.subheader("5 \u00b7 On-Chain Live State")

if onchain_f.empty:
    st.info("No on-chain data for selected chains.")
else:
    top_liq = onchain_f.nlargest(15, "liquidity")
    fig = px.bar(
        top_liq.sort_values("liquidity"),
        x="liquidity", y="label", orientation="h",
        color="chain", color_discrete_map=CHAIN_COLORS,
        labels={"liquidity": "Active Liquidity (raw units)", "label": ""},
        title="Live Active Liquidity by Pool (Top 15)",
    )
    fig.update_xaxes(tickformat=".2e")
    fig.update_layout(height=500)
    st.plotly_chart(fig, use_container_width=True)

    with st.expander("Full on-chain table"):
        display_cols = [c for c in ["label", "chain", "price", "tick", "liquidity", "fetched_at"] if c in onchain_f.columns]
        st.dataframe(
            onchain_f[display_cols].sort_values("liquidity", ascending=False).reset_index(drop=True)
            .style.format({"liquidity": "{:.2e}", "price": "{:.6f}"}),
            use_container_width=True,
        )

st.divider()
st.caption("Data: DeFiLlama \u00b7 Dune Analytics \u00b7 Uniswap V3 on-chain ABIs \u00b7 Built with Streamlit")

2026-04-05 00:22:05.169 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:05.170 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:05.171 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:05.240 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-04-05 00:22:05.245 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:05.246 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 00:22:05.247 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()

In [11]:
# Cell 10 — Launch dashboard
# Converts this notebook to a temp script, then opens a new terminal running Streamlit.
# A new cmd window will appear — wait for "You can now view your Streamlit app" before opening the link.
import subprocess, pathlib, os

cwd      = pathlib.Path.cwd()
nb_path  = cwd / "dashboard.ipynb"
out_stem = cwd / ".dashboard_run"
out_py   = cwd / ".dashboard_run.py"

subprocess.run(
    ["jupyter", "nbconvert", "--to", "python", str(nb_path), "--output", str(out_stem)],
    check=True
)

assert out_py.exists(), f"nbconvert did not produce {out_py}"

os.system(f"start cmd /k "streamlit run \"{out_py}\""")
print("Streamlit is starting in a new terminal window.")
print("Once you see 'You can now view your Streamlit app', open: http://localhost:8501")

AssertionError: nbconvert did not produce C:\Users\Zhou\FinTech-590\DeFi-Liquidity-Pool Project\.dashboard_run.py